# PRISM OOD - CONCH
## Cross-Dataset Transfer: 4 Pairs
- PCam→MHIST, MHIST→PCam, CRC→BRACS, BRACS→CRC

In [ ]:
import torch, os, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from scipy.optimize import minimize_scalar
from PIL import Image
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/PRISM/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS = [42, 123, 456]
MKEY = 'conch'
print(f"Device: {DEVICE} | Model: CONCH")

In [ ]:
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
from huggingface_hub import login
login(token=HF_TOKEN)

from conch.open_clip_custom import create_model_from_pretrained
model, _ = create_model_from_pretrained('conch_ViT-B-16', "hf_hub:MahmoodLab/conch")
model = model.to(DEVICE)
model.eval()
def forward_model(m, x):
    return m.encode_image(x, proj_contrast=False, normalize=True)
print("CONCH loaded")

In [ ]:
def extract_features(model, dataloader):
    model.eval()
    features, labels = [], []
    with torch.no_grad():
        for imgs, lbls in dataloader:
            imgs = imgs.to(DEVICE)
            feats = forward_model(model, imgs)
            features.append(feats.cpu().numpy())
            labels.append(lbls.numpy() if hasattr(lbls, 'numpy') else np.array(lbls))
    return np.vstack(features), np.concatenate(labels)

def compute_ece(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() > 0:
            ece += mask.mean() * abs(labels[mask].mean() - probs[mask].mean())
    return ece

def temperature_scale_binary(logits, labels):
    def nll(T):
        p = 1 / (1 + np.exp(-logits / T))
        p = np.clip(p, 1e-7, 1-1e-7)
        return -np.mean(labels * np.log(p) + (1-labels) * np.log(1-p))
    res = minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded')
    return res.x

def stratified_sample(indices, labels, fraction, seed):
    np.random.seed(seed)
    classes = np.unique(labels)
    sampled = []
    for c in classes:
        c_idx = indices[labels == c]
        n = max(1, int(len(c_idx) * fraction))
        sampled.extend(np.random.choice(c_idx, size=n, replace=False))
    return np.array(sampled)

print("Helper functions ready!")

In [ ]:
from torchvision.datasets import PCAM
import csv

Image.MAX_IMAGE_PIXELS = None
TRANSFORM = transforms.Compose([
    transforms.Resize(224), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

PCAM_DIR  = '/content/drive/MyDrive/PRISM/datasets/pcam'
MHIST_DIR = '/content/drive/MyDrive/PRISM/datasets/mhist'
CRC_DIR   = '/content/drive/MyDrive/PRISM/datasets/crc/NCT-CRC-HE-100K'
BRACS_DIR = '/content/drive/MyDrive/PRISM/datasets/bracs/latest_version'

class PCamDataset(Dataset):
    def __init__(self, split='train'):
        self.ds = PCAM(root=PCAM_DIR, split=split, transform=TRANSFORM, download=False)
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        img, lbl = self.ds[i]; return img, int(lbl)

class MHISTDataset(Dataset):
    def __init__(self, split='train'):
        self.samples = []
        with open(os.path.join(MHIST_DIR, 'annotations.csv')) as f:
            for row in csv.DictReader(f):
                if row['Partition'].lower() == split.lower():
                    p = os.path.join(MHIST_DIR, 'images', row['Image Name'])
                    l = 0 if row['Majority Vote Label'] == 'HP' else 1
                    self.samples.append((p, l))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, l = self.samples[i]
        return TRANSFORM(Image.open(p).convert('RGB')), l

class CRCBinaryDataset(Dataset):
    def __init__(self, root=CRC_DIR):
        self.samples = []
        for cls in sorted(os.listdir(root)):
            d = os.path.join(root, cls)
            if not os.path.isdir(d): continue
            lbl = 1 if cls == 'TUM' else 0
            for f in os.listdir(d):
                if f.lower().endswith(('.png','.jpg','.tif')):
                    self.samples.append((os.path.join(d, f), lbl))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, l = self.samples[i]
        return TRANSFORM(Image.open(p).convert('RGB')), l

BRACS_MALIGNANT = {'IC', 'DCIS'}
class BRACSBinaryDataset(Dataset):
    def __init__(self, split='train'):
        self.samples = []
        split_dir = os.path.join(BRACS_DIR, split)
        for cls in sorted(os.listdir(split_dir)):
            d = os.path.join(split_dir, cls)
            if not os.path.isdir(d): continue
            lbl = 1 if cls in BRACS_MALIGNANT else 0
            for f in os.listdir(d):
                if f.lower().endswith(('.png','.jpg','.tif')):
                    self.samples.append((os.path.join(d, f), lbl))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, l = self.samples[i]
        return TRANSFORM(Image.open(p).convert('RGB')), l

print("Datasets defined!")

In [ ]:
def run_ood(src_ds, src_name, tgt_ds, tgt_name):
    print(f"\n{'='*55}\nOOD: {src_name} → {tgt_name}\n{'='*55}")
    
    src_loader = DataLoader(src_ds, batch_size=256, num_workers=2, pin_memory=True)
    tgt_loader = DataLoader(tgt_ds, batch_size=256, num_workers=2, pin_memory=True)
    
    print("Extracting source features...")
    src_feats, src_labels = extract_features(model, src_loader)
    print("Extracting target features...")
    tgt_feats, tgt_labels = extract_features(model, tgt_loader)
    print(f"Src: {src_feats.shape} | Tgt: {tgt_feats.shape}")
    print(f"Src label dist: {np.bincount(src_labels.astype(int))}")
    print(f"Tgt label dist: {np.bincount(tgt_labels.astype(int))}")
    
    src_idx = np.arange(len(src_labels))
    results, ts_results = [], []
    
    for frac in LABEL_FRACTIONS:
        for seed in SEEDS:
            sampled = stratified_sample(src_idx, src_labels.astype(int), frac, seed)
            clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
            clf.fit(src_feats[sampled], src_labels[sampled].astype(int))
            
            logits = clf.decision_function(tgt_feats)
            probs  = 1 / (1 + np.exp(-logits))
            preds  = (probs > 0.5).astype(int)
            tgt_int = tgt_labels.astype(int)
            
            try:    auroc = roc_auc_score(tgt_int, probs)
            except: auroc = float('nan')
            ece_raw = compute_ece(probs, tgt_int)
            brier   = brier_score_loss(tgt_int, probs)
            f1      = f1_score(tgt_int, preds, average='macro', zero_division=0)
            
            try:
                T = temperature_scale_binary(logits, tgt_int)
                sp = 1 / (1 + np.exp(-logits / T))
                ece_sc = compute_ece(sp, tgt_int)
                ece_imp = ece_raw - ece_sc
            except:
                T = ece_sc = ece_imp = float('nan')
            
            results.append({'fraction':frac,'seed':seed,'auroc':auroc,'ece':ece_raw,'f1':f1,'brier':brier})
            ts_results.append({'fraction':frac,'seed':seed,'temperature':T,'ece_raw':ece_raw,'ece_scaled':ece_sc,'ece_improvement':ece_imp})
            print(f"  {frac:.0%} s{seed}: AUROC={auroc:.4f} ECE={ece_raw:.4f} F1={f1:.4f}")
    
    tag = f"{MKEY}_ood_{src_name.lower()}_to_{tgt_name.lower()}"
    pd.DataFrame(results).to_csv(f'{RESULTS_DIR}/{tag}_results.csv', index=False)
    pd.DataFrame(ts_results).to_csv(f'{RESULTS_DIR}/{tag}_temperature_scaling.csv', index=False)
    print(f"✅ Saved: {tag}")
    return pd.DataFrame(results)

print("run_ood() defined!")

In [ ]:
print("Loading all datasets...")
pcam_train  = PCamDataset('train')
pcam_test   = PCamDataset('test')
mhist_train = MHISTDataset('train')
mhist_test  = MHISTDataset('test')
crc_all     = CRCBinaryDataset()
bracs_train = BRACSBinaryDataset('train')
bracs_test  = BRACSBinaryDataset('test')

# CRC 70/15/15 split
crc_idx = np.arange(len(crc_all))
np.random.seed(42); np.random.shuffle(crc_idx)
crc_train = Subset(crc_all, crc_idx[:int(0.70*len(crc_all))])
crc_test  = Subset(crc_all, crc_idx[int(0.85*len(crc_all)):])

print(f"PCam: {len(pcam_train)}/{len(pcam_test)}")
print(f"MHIST: {len(mhist_train)}/{len(mhist_test)}")
print(f"CRC binary: {len(crc_train)}/{len(crc_test)}")
print(f"BRACS binary: {len(bracs_train)}/{len(bracs_test)}")

# Run all 4 OOD pairs
r1 = run_ood(pcam_train,  'PCam',  mhist_test,  'MHIST')
r2 = run_ood(mhist_train, 'MHIST', pcam_test,   'PCam')
r3 = run_ood(crc_train,   'CRC',   bracs_test,  'BRACS')
r4 = run_ood(bracs_train, 'BRACS', crc_test,    'CRC')

print("\n🎉 ALL OOD EXPERIMENTS DONE!")

In [ ]:
for name, r in [('PCam→MHIST',r1),('MHIST→PCam',r2),('CRC→BRACS',r3),('BRACS→CRC',r4)]:
    s = r.groupby('fraction')[['auroc','ece','f1']].mean()
    print(f"\n{name}:")
    print(s.round(4).to_string())